In [ ]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub')
    print("Setup complete!")


In [1]:
input_dir = 'datasets/preprocessed'

In [2]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import csv
import glob

print(f"Checking datasets in {input_dir}...")
if not os.path.exists(input_dir):
    print(f"Error: {input_dir} not found.")
else:
    required_keys = {"text", "label"}
    
    dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
    for input_file in dataset_files:
        print(f"\n--- Checking {os.path.basename(input_file)} ---")
        labels_found = set()
        error_records = []

        line_num = 0
        with open(input_file, "r", encoding="utf-8") as f:
            for line in f:
                line_num += 1
                error_msg = None
                try:
                    record = json.loads(line)
                    if not required_keys.issubset(record.keys()):
                        error_msg = f"Missing required keys. Found: {list(record.keys())}"
                    elif not record.get("label"):
                        error_msg = "Empty label field."
                    elif not record.get("text") or not str(record.get("text")).strip():
                        error_msg = "Empty text field."
                    else:
                        labels_found.add(record.get("label"))

                    if error_msg:
                        error_records.append({
                            "line_num": line_num,
                            "error": error_msg,
                            "raw_line": line.strip()
                        })
                except json.JSONDecodeError:
                    error_records.append({
                        "line_num": line_num,
                        "error": "Invalid JSON.",
                        "raw_line": line.strip()
                    })

        if len(error_records) == 0:
            print(f"Dataset check passed! Validated {line_num} records.")
            print(f"Found {len(labels_found)} unique labels, e.g., {list(labels_found)[:5]}")
        else:
            print(f"Dataset check failed with {len(error_records)} errors.")

            # Save errors to CSV
            output_csv = input_file.replace(".jsonl", "_errors.csv")
            with open(output_csv, "w", encoding="utf-8", newline="") as csvfile:
                fieldnames = ["line_num", "error", "raw_line"]
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                for er in error_records:
                    writer.writerow(er)
            print(f"Exported error details to {output_csv}")


Checking datasets in datasets/preprocessed...

--- Checking flores_plus.jsonl ---
Dataset check passed! Validated 223652 records.
Found 206 unique labels, e.g., ['vec', 'sun', 'dzo', 'khm', 'ars']

--- Checking commonlid.jsonl ---
Dataset check passed! Validated 373230 records.
Found 109 unique labels, e.g., ['vec', 'fro', 'ars', 'urd', 'aeb']

--- Checking wili-2018.jsonl ---
Dataset check passed! Validated 235000 records.
Found 235 unique labels, e.g., ['vec', 'sun', 'sgs', 'ori', 'sme']


In [3]:
import joblib
import pandas as pd

MODEL_DIR = "../models"
SINHALA_LABEL = "sin"
MODEL_LABEL_FOR_SINHALA = "sinhala"

vectorizer_path = os.path.join(MODEL_DIR, "langid_vectorizer.pkl")
clf_path = os.path.join(MODEL_DIR, "langid_model.pkl")

all_mismatches = []

if not os.path.exists(vectorizer_path) or not os.path.exists(clf_path):
    print("\nBaseline model files not found. Skipping baseline model check.")
else:
    vectorizer = joblib.load(vectorizer_path)
    clf = joblib.load(clf_path)
    
    dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
    for input_file in dataset_files:
        sinhala_records = []
        with open(input_file, "r", encoding="utf-8") as f:
            for line in f:
                record = json.loads(line)
                if record.get("label") == SINHALA_LABEL:
                    sinhala_records.append(record)

        if not sinhala_records:
            print(f"\nNo '{SINHALA_LABEL}'-labeled records in {os.path.basename(input_file)}; skipping baseline check.")
        else:
            sinhala_df = pd.DataFrame(sinhala_records)
            X = vectorizer.transform(sinhala_df["text"])
            sinhala_df["predicted_label"] = clf.predict(X)

            mismatches = sinhala_df[sinhala_df["predicted_label"] != MODEL_LABEL_FOR_SINHALA].reset_index(drop=True)

            print(f"\nBaseline model check on '{SINHALA_LABEL}' rows in {os.path.basename(input_file)}:")
            print(f"  {len(sinhala_df)} rows labeled '{SINHALA_LABEL}', {len(mismatches)} not predicted as '{MODEL_LABEL_FOR_SINHALA}'")

            if not mismatches.empty:
                print(mismatches["predicted_label"].value_counts().to_string())

                checks_dir = "datasets/checks"
                os.makedirs(checks_dir, exist_ok=True)
                dataset_name = os.path.splitext(os.path.basename(input_file))[0]
                mismatches_path = os.path.join(checks_dir, f"{dataset_name}_sinhala_mismatches.csv")
                mismatches.to_csv(mismatches_path, index=False)
                print(f"Saved mismatches to {mismatches_path}")
                all_mismatches.append(mismatches)
                
if all_mismatches:
    final_mismatches = pd.concat(all_mismatches, ignore_index=True)
else:
    final_mismatches = pd.DataFrame(columns=["text", "label", "source", "predicted_label"])

final_mismatches


/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-li


Baseline model check on 'sin' rows in flores_plus.jsonl:
  1012 rows labeled 'sin', 1 not predicted as 'sinhala'
predicted_label
sanskrit    1
Saved mismatches to datasets/checks/flores_plus_sinhala_mismatches.csv

No 'sin'-labeled records in commonlid.jsonl; skipping baseline check.

Baseline model check on 'sin' rows in wili-2018.jsonl:
  1000 rows labeled 'sin', 2 not predicted as 'sinhala'
predicted_label
sanskrit    1
pali        1
Saved mismatches to datasets/checks/wili-2018_sinhala_mismatches.csv


,text,label,source,predicted_label
0,මැස්ලෝගේ අවශ්‍යතා ධූරාවලි න්‍යාය සහ හර්ට්ස්බර්...,sin,flores_plus,sanskrit
1,නජාට්‌ වැලවුඩ් බෙල්කේසම් (උපත 1977 ඔක්තෝම්බර් ...,sin,wili-2018,sanskrit
2,කායානුපස්සනා කර්මස්ථානයේ හඳුනාගත යුතු කොටස් කි...,sin,wili-2018,pali
